In [1]:
!pip install unsloth datasets
!pip install trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 11.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.5 MB/s eta 0:00:00
   

In [2]:
import torch 
import random

print(f"Pytorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Pytorch: 2.10.0+cu128
CUDA: True


In [3]:
from unsloth import FastLanguageModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model)

print(f"[OK] Loaded {BASE_MODEL}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[OK] Loaded Qwen/Qwen2.5-1.5B-Instruct


In [4]:
def generate_text(model,tokenizer,prompt,max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids = inputs.input_ids,
            attention_mask = inputs.attention_mask,
            max_new_tokens = max_new_tokens,
            use_cache = True,
            repetition_penalty = 1.2,
            do_sample = True,
            temperature = 0.7
        )
    new_tokens = outputs[0,inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [5]:
from datasets import load_dataset

pref_dataset = load_dataset(
    "argilla/ultrafeedback-binarized-preferences-cleaned",
    split="train"
)

print(f"[OK] Loaded {len(pref_dataset)} preference pairs")
print(f"Columns: {pref_dataset.column_names}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60917 [00:00<?, ? examples/s]

[OK] Loaded 60917 preference pairs
Columns: ['source', 'prompt', 'chosen', 'chosen-rating', 'chosen-model', 'rejected', 'rejected-rating', 'rejected-model']


In [6]:
for i in range(3):
    print(pref_dataset[i])

{'source': 'evol_instruct', 'prompt': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}', 'chosen': [{'content': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}', 'role': 'user'}, 

Format the Dataset for DPOtrainer

In [7]:
def format_for_dpo(example):
    prompt = example.get("prompt", "")
    chosen = example.get("chosen", "")
    rejected = example.get("rejected", "")

    if isinstance(prompt,list):
        prompt = prompt[0].get("content", str(prompt[0])) if prompt else ""
    if isinstance(chosen,list):
        chosen = chosen[-1].get("content", str(chosen[-1])) if chosen else ""
    if isinstance(rejected,list):
        rejected = rejected[-1].get("content", str(rejected[-1])) if rejected else ""

    return {
        "prompt": str(prompt),
        "chosen": str(chosen),
        "rejected": str(rejected)
    }

formatted = pref_dataset.map(format_for_dpo, remove_columns=pref_dataset.column_names)

formatted = formatted.shuffle(seed=SEED).select(range(min(3000,len(formatted))))

split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_data = split["train"]
test_data = split["test"]

print(f"Train: {len(train_data)} | Eval: {len(test_data)}")
s = train_data[0]
print(f"Prompt: {s['prompt']}")
print(f"Chosen: {s['chosen']}")
print(f"Rejected: {s['rejected']}")

Map:   0%|          | 0/60917 [00:00<?, ? examples/s]

Train: 2850 | Eval: 150
Prompt: Given a premise and two alternatives in Hindi, choose the alternative that is either a plausible cause or effect of the situation described by the premise. The premise is the 'कथन' field and the alternatives are the 'विकल्प A' and 'विकल्प B' fields. The output should either be "विकल्प A" or "विकल्प B" based on your judgment.
Q: कथन: स्कीयर ढलान पर फिसल गया।

 विकल्प A: उसने अपना स्की पोल गिरा दिया।

 विकल्प B: उसने बर्फ का एक टुकड़ा मारा।
A: 
Chosen: विकल्प A
Rejected: विकल्प A: उसने अपना स्की पोल गिरा दिया।

Confidence: 85%


DPO training

In [8]:
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True
)

# Adding LoRa cofig
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable= {trainable} / {total}")
print(f"Percentage: {trainable/total * 100:.2f}")

==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable= 18464768 / 1036449280
Percentage: 1.78


In [9]:
from unsloth import PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_length=512,
    max_prompt_length=256,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2, # keep most recent 2 checkpoints
    # fallback to fp16 if bf16 is not availlable
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=SEED,
    report_to="none",
)

print("[OK] DPO config ready")
print(f"Learning rate: {dpo_config.learning_rate} (compare SFT: 2e-4)")
print(f"Beta: {dpo_config.beta}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[OK] DPO config ready
Learning rate: 5e-06 (compare SFT: 2e-4)
Beta: 0.1


In [10]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
)

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

In [11]:
dpo_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,850 | Num Epochs = 1 | Total steps = 179
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
50,0.688928,0.689004,0.002908,-0.006035,0.657895,0.008943,-413.720520,-316.088287,-0.434339,-0.423595
100,0.684430,0.681495,0.012106,-0.012862,0.677632,0.024968,-413.628540,-316.156555,-0.439869,-0.431806
150,0.684633,0.678769,0.014200,-0.016844,0.677632,0.031044,-413.607574,-316.196381,-0.439478,-0.431751
179,0.677932,0.679615,0.012884,-0.016358,0.684211,0.029242,-413.620758,-316.191498,-0.439374,-0.431657


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=179, training_loss=0.6853794785185233, metrics={'train_runtime': 2915.6689, 'train_samples_per_second': 0.977, 'train_steps_per_second': 0.061, 'total_flos': 0.0, 'train_loss': 0.6853794785185233, 'epoch': 1.0})

In [12]:
FastLanguageModel.for_inference(model)

# Test: Generate responses and see if quality improved
test_prompts = [
    "Explain what inflation is in simple terms.",
    "What should I consider before investing in stocks?",
    "Help, I'm stressed about a job interview tomorrow.",
    "What is the difference between a bond and a stock?",
]

print("=" * 70)
print("AFTER DPO -- Generation Quality Test")
print("=" * 70)
for prompt in test_prompts:
    response = generate_text(model, tokenizer, prompt, max_new_tokens=150)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:400]}")
    print("-" * 50)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER DPO -- Generation Quality Test


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Q: Explain what inflation is in simple terms.
A:  Inflation refers to the general increase in prices of goods and services over time, which leads to a decrease in purchasing power or money's worth.

For example:

1) If you buy something for $50 today but it costs more than that when measured against other items 6 months later.
2) Your wages do not keep up with rising cost of living (e.g., housing, food).
3) More people can afford products becaus
--------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What should I consider before investing in stocks?
A:  Investing in the stock market can be a lucrative strategy, but it's important to understand that there are risks involved. Here are some key factors you should consider:

1. Risk tolerance: Determine how much risk you're willing to take on with your investment.

2. Time horizon: How long do you plan to hold onto an asset or portfolio?

3. Investment objective(s): Do you want capital appreciation 
--------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Help, I'm stressed about a job interview tomorrow.
A:  Can you give me some advice?
Certainly! Here are several tips to help reduce your stress and increase the chances of success in your upcoming job interview:

1. **Prepare Thoroughly**: Review all the information that will be discussed during the interview. Make sure you understand what is expected from each question.

2. **Practice Your Answers**: Rehearse common questions or scenarios with frien
--------------------------------------------------

Q: What is the difference between a bond and a stock?
A:  Bonds are debt instruments that represent an agreement to pay back borrowed money at some point in the future. Stocks, on the other hand, represent ownership of part of a company.

Bonds typically offer fixed interest payments over time while stocks allow investors to own shares of a particular corporation or organization. The value of bonds can fluctuate based on market conditions but their prin
-------------------------------

In [13]:
import torch.nn.functional as F

FastLanguageModel.for_inference(model)

def compute_response_logprob(model, tokenizer, prompt, response):
    full_text = prompt + " " + response
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(model.device)
    prompt_len = prompt_ids.shape[1]

    with torch.no_grad():
        outputs = model(input_ids=full_ids)
        logits = outputs.logits

    shift_logits = logits[0, prompt_len-1:-1, :]
    shift_labels = full_ids[0, prompt_len:]

    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(1, shift_labels.unsqueeze(1)).squeeze(1)
    return token_log_probs.mean().item()

# Use test_data (your eval split), not eval_data
correct = 0
total_test = min(20, len(test_data))

for i in range(total_test):
    ex = test_data[i]
    chosen_lp = compute_response_logprob(model, tokenizer, ex["prompt"], ex["chosen"])
    rejected_lp = compute_response_logprob(model, tokenizer, ex["prompt"], ex["rejected"])
    prefers_chosen = chosen_lp > rejected_lp
    if prefers_chosen:
        correct += 1
    if i < 5:
        status = "✅" if prefers_chosen else "❌"
        print(f"{status} Example {i+1}: chosen={chosen_lp:.4f} rejected={rejected_lp:.4f} margin={chosen_lp-rejected_lp:.4f}")

print(f"\nPreference accuracy: {correct}/{total_test} = {100*correct/total_test:.1f}%")


✅ Example 1: chosen=-1.7584 rejected=-1.7713 margin=0.0129
✅ Example 2: chosen=-0.9073 rejected=-0.9618 margin=0.0545
❌ Example 3: chosen=-0.9226 rejected=-0.7423 margin=-0.1803
✅ Example 4: chosen=-0.5471 rejected=-2.3548 margin=1.8077
✅ Example 5: chosen=-1.1685 rejected=-1.4449 margin=0.2764

Preference accuracy: 15/20 = 75.0%
